In [ ]:
from pathlib import Path
import ixmp4
import pyam
import nomenclature

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/NGFS/").iterdir())
    ]
)

In [ ]:
# rename scenarios for clear reference to ENGAGE
df.rename(
    scenario=dict([(i, "NGFS Phase 5-" + i) for i in df.scenario]),
    inplace=True,
)

In [ ]:
df.rename(
    region=dict([(i, i.replace(" NGFS", "")) for i in df.filter(region="GCAM*").region]),
    inplace=True,
)

In [ ]:
scenario_mapping = {
    "NGFS Phase 5-Net Zero 2050": "NGFS Phase 5-Net-Zero 2050",
}
df.rename(scenario=scenario_mapping, inplace=True)

In [ ]:
#definition = nomenclature.DataStructureDefinition("../definitions/")
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
platform = ixmp4.Platform("scenariocompass-dev")

In [ ]:
platform.runs.tabulate(scenario="NGFS*")

In [ ]:
df.rename(
    region={
        "GCAM 6.0|Australia_NZ": "GCAM 6.0|Australia and New Zealand",
        "MESSAGEix-GLOBIOM 2.0-R12|Rest Centrally Planned Asia": "MESSAGEix-GLOBIOM 2.0-R12|Rest of Centrally Planned Asia",
        "MESSAGEix-GLOBIOM 2.0-R12|Sub-saharan Africa": "MESSAGEix-GLOBIOM 2.0-R12|Sub-Saharan Africa",
        "REMIND-MAgPIE 3.3-4.8|Canada, NZ, Australia": "REMIND-MAgPIE 3.3-4.8|Canada, Australia, New Zealand",
        "REMIND-MAgPIE 3.3-4.8|China": "REMIND-MAgPIE 3.3-4.8|China and Taiwan",
        "REMIND-MAgPIE 3.3-4.8|Countries from the Reforming Economies of the Former Soviet Union": "REMIND-MAgPIE 3.3-4.8|Russia and Reforming Economies",
        "REMIND-MAgPIE 3.3-4.8|Middle East, North Africa, Central Asia": "REMIND-MAgPIE 3.3-4.8|Middle East and North Africa",
        "REMIND-MAgPIE 3.3-4.8|Sub-saharan Africa": "REMIND-MAgPIE 3.3-4.8|Sub-Saharan Africa",
    },
    inplace=True,
)

In [ ]:
project = ["engage", "ngfs5"]
legacy_mapping = {}


for _project in project:
    for code, attrs in definition.variable.items():
        if _project in attrs.extra_attributes:
            legacy_mapping[attrs.__getattr__(_project)] = code
    
    df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "km3/yr": "million m3/yr",
        "US$2010/GJ": "USD_2010/GJ",
    },
    inplace=True,
)

In [ ]:
df.rename(
    variable=dict(
        [
            (
                i,
                i.replace("Residential and Commercial|Commercial", "Commercial").replace("Residential and Commercial|Residential", "Residential")
            )
            for i in df.filter(variable="*Residential and Commercial|*").variable
        ]
    ),
    inplace=True,
)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR6 climate diagnostics*",
        "Capacity Additions|Electricity|Storage Capacity",
        "Capacity|Electricity|Storage",
        "*including medium chronic physical risk damage estimate",
        "*Average 2016-2030",
        "Post-processed*",
        "Carbon Sequestration|CCS", ## may need to be moved
        "Carbon Sequestration|CCS|Biomass|Energy|Supply|Electricity",
        "Carbon Sequestration|CCS|Biomass|Energy|Supply|Hydrogen", 
        "Carbon Sequestration|CCS|Biomass|Energy|Supply|Liquids",
        "Carbon Sequestration|CCS|Biomass|Energy|Demand|Industry",
        "Carbon Sequestration|CCS|Fossil|Energy|Supply|Electricity",
        "Carbon Sequestration|CCS|Fossil|Energy|Supply|Hydrogen",
        "Emissions|CO2|Energy|Demand|Industry|*",
        "GDP|MER|Counterfactual without damage",
        "GDP|PPP|Counterfactual without damage",
        "Investment|Energy Supply|CO2 Transport and Storage",
        "Forcing",
        "Carbon Sequestration|CCS|Fossil|Energy|Supply|Liquids",
        "Agricultural Production|Energy", ## this is a duplicate
        "Secondary Energy",
        "Revenue|Government|Tax|Carbon*",
        "Agricultural Production|Energy|Residues",
        "Agricultural Production|Non-Energy",
        "Agricultural Production|Non-Energy|Crops",
        "Agricultural Production|Non-Energy|Livestock",
        "Price|Carbon|Demand|Industry",
        "Price|Carbon|Demand|Residential and Commercial",
        "Price|Carbon|Demand|Transportation",
        "Price|Carbon|Supply",
        "Price|*|Index",
        "Water Withdrawal|Irrigation", ## there is confusion about the unit
    ],
    keep=False,
    inplace=True
)

In [ ]:
# update carbon-management variables
variable_mapping = {
    "Agricultural Demand|Crops|Energy": "Agricultural Demand|Crops|Bioenergy",
    "Agricultural Demand|Crops|Energy|1st generation": "Agricultural Demand|Crops|Bioenergy|1st Generation",
    "Agricultural Demand|Crops|Energy|2nd generation": "Agricultural Demand|Crops|Bioenergy|2nd Generation",
#    "Carbon Sequestration|CCS": "Carbon Removal|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
    "Yield|Cereal": "Yield|Cropland|Cereals",
    "Yield|Oilcrops": "Yield|Cropland|Oil Crops",
    "Yield|Sugarcrops": "Yield|Cropland|Sugar Crops",
    "Secondary Energy|Gases|Natural Gas": "Secondary Energy|Gases|Gas"
}

df.rename(variable=variable_mapping, inplace=True)

In [ ]:
validation_args = ["upper_bound", "lower_bound", "value", "rtol", "atol", "range"]

validation_list = list()

for name, variable in definition.variable.items():
    if any([i in validation_args for i in variable.extra_attributes]):
        validation_list.append(
            dict(
                variable=name,
                validation=[dict([(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])]
            )
        )

In [ ]:
validator = nomenclature.processor.DataValidator(criteria_items=validation_list, file=".")

In [ ]:
validator.apply(df)

In [ ]:
df._data.loc[(
    "MESSAGEix-GLOBIOM 2.0-M-R12-NGFS",
    "NGFS Phase 5-Delayed Transition",
    "MESSAGEix-GLOBIOM 2.0-R12|Rest of Centrally Planned Asia", "Carbon Capture|Energy|Supply|Fossil", "Mt CO2/yr", 2060)
] = 0

In [ ]:
df.filter(variable=definition.variable, inplace=True)

In [ ]:
definition.validate(df)

In [ ]:
region_processor = nomenclature.RegionProcessor.from_directory("../../common-definitions/mappings/", definition)

In [ ]:
df = region_processor.apply(region_processor.revert(df))

In [ ]:
df_public = pyam.read_ixmp4(platform, scenario="NGFS*")

In [ ]:
[v for v in df.variable if v not in df_public.variable]

In [ ]:
[v for v in df_public.variable if v not in df.variable]

In [ ]:
filter_args = dict(model="REMIND-MAgPIE 3.3-4.8", scenario="NGFS Phase 5-Low Demand")

pyam.compare(
    df.filter(region=["*(R9)"], keep=False).filter(year=range(2020, 2101, 5), **filter_args),
    df_public.filter(**filter_args),
).reset_index().to_excel( "foo.xlsx") 

In [ ]:
df.filter(variable="Capacity|Hydrogen", model="REMIND-MAgPIE 3.3-4.8", scenario="NGFS Phase 5-Low Demand").timeseries()

In [ ]:
for model, scenario in df.index:
    _df = df.filter(model=model, scenario=scenario)
    run = platform.runs.get(model=model, scenario=scenario)
    run.iamc.add(_df.data)
    print(f"Imported {model} - {scenario}")

In [ ]:
definition.validate(df)

In [ ]:
platform = ixmp4.Platform("scenariocompass-migration")

In [ ]:
df.scenario

In [ ]:
platform.runs.tabulate(scenario="NGFS*", default_only=False)

In [ ]:
df_remind = df.filter(model="REMIND-MAgPIE 3.3-4.8 - Integrated Physical Damages (median)")

In [ ]:
df_remind.region

In [ ]:
run.iamc.tabulate()

In [ ]:
run = platform.runs.get("MESSAGEix-GLOBIOM 2.0-M-R12-NGFS", "NGFS Phase 5-Current Policies", version=1)
_df = df.filter(model=run.model.name, scenario=run.scenario.name)
run.iamc.add(_df.data)
run.meta = dict(_df.meta.loc[(run.model.name, run.scenario.name)])
run.set_as_default()

In [ ]:
for run in runs:
    data = run.iamc.tabulate(variable="Carbon Removal|Geological Storage")
    if not data.empty:
        run.iamc.remove(data)

In [ ]:
runs = platform.runs.list(default_only=False, is_default=False)

In [ ]:
runs

In [ ]:
for run in runs:
    _df = df.filter(model=run.model.name, scenario=run.scenario.name)
    run.iamc.add(_df.data)
    run.meta = dict(_df.meta.loc[(run.model.name, run.scenario.name)])
    run.set_as_default()

In [ ]:
for model, scenario in df.index:
    try:
        platform.runs.get(model=model, scenario=scenario)
    except:
        df.filter(model=model, scenario=scenario).to_ixmp4("scenariocompass-migration")